In [56]:
import pandas as pd

In [57]:
visits = pd.read_csv('visits.csv')
registrations = pd.read_csv('registrations.csv')
ads = pd.read_csv('ads.csv')

In [58]:
visits.head()

,uuid,platform,user_agent,date
0,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-06-01T11:12:38
1,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-05-30T13:44:46
2,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-05-29T04:37:54
3,0919c209-9aa2-4c11-ba6f-6a15669954cf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-06-01T05:58:14
4,0919c209-9aa2-4c11-ba6f-6a15669954cf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-05-29T01:22:31


In [59]:
registrations.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T07:40:13,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ebyrd@example.org,web,google
1,2023-03-01T13:14:00,f007f97c-9d8b-48b5-af08-119bb8f6d9b6,knightgerald@example.org,web,email
2,2023-03-01T03:05:50,24ff46ae-32b3-4a74-8f27-7cf0b8f32f15,cherylthompson@example.com,web,apple
3,2023-03-01T00:04:47,3e9914e1-5d73-4c23-b25d-b59a3aeb2b60,halldavid@example.org,web,email
4,2023-03-01T18:31:52,27f875fc-f8ce-4aeb-8722-0ecb283d0760,denise86@example.net,web,google


In [60]:
ads.head()

,date,utm_source,utm_medium,utm_campaign,cost
0,2023-03-01T10:54:41,google,cpc,advanced_algorithms_series,212
1,2023-03-02T10:32:35,google,cpc,advanced_algorithms_series,252
2,2023-03-03T19:21:40,google,cpc,advanced_algorithms_series,202
3,2023-03-04T17:52:04,google,cpc,advanced_algorithms_series,223
4,2023-03-05T05:35:13,google,cpc,advanced_algorithms_series,265


In [63]:
visits = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)].copy()

visits['date'] = pd.to_datetime(visits['date'])
registrations['date'] = pd.to_datetime(registrations['date'])
ads['date'] = pd.to_datetime(ads['date'])

visits['date_group'] = visits['date'].dt.date
registrations['date_group'] = registrations['date'].dt.date
ads['date_group'] = ads['date'].dt.date

visits_grouped = (
    visits
    .groupby('date_group')
    .size()
    .reset_index(name='visits')
)

registrations_grouped = (
    registrations
    .groupby('date_group')
    .size()
    .reset_index(name='registrations')
)

base_df = pd.merge(
    visits_grouped,
    registrations_grouped,
    on='date_group',
    how='outer'
)

base_df['visits'] = base_df['visits'].fillna(0)
base_df['registrations'] = base_df['registrations'].fillna(0)

ads_grouped = (
    ads
    .groupby(['date_group', 'utm_campaign'], as_index=False)
    .agg({'cost': 'sum'})
)

result = pd.merge(
    base_df,
    ads_grouped,
    on='date_group',
    how='left'
)

result['cost'] = result['cost'].fillna(0)
result['utm_campaign'] = result['utm_campaign'].fillna('none')

result = result.sort_values('date_group').reset_index(drop=True)

result['date_group'] = result['date_group'].astype(str)
result.to_json('ads.json')

In [64]:
result

,date_group,visits,registrations,utm_campaign,cost
0,2023-03-01,941,87,advanced_algorithms_series,212.0
1,2023-03-02,1226,106,advanced_algorithms_series,252.0
2,2023-03-03,1564,107,advanced_algorithms_series,202.0
3,2023-03-04,1754,159,advanced_algorithms_series,223.0
4,2023-03-05,1838,115,advanced_algorithms_series,265.0
...,...,...,...,...,...
179,2023-08-27,1601,88,intro_to_python_course,222.0
180,2023-08-28,1337,83,intro_to_python_course,223.0
181,2023-08-29,1679,143,intro_to_python_course,190.0
182,2023-08-30,1458,101,intro_to_python_course,109.0


In [65]:
result = result.sort_values('date_group')
result['date_group'] = result['date_group'].astype(str)
result.to_json('ads.json')